<a href="https://colab.research.google.com/github/nitheshl72/localLLM/blob/main/local_llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Sat Jan 31 23:49:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip install -U pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 89.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [ ]:
from huggingface_hub import list_repo_files

files = list_repo_files("Qwen/Qwen2.5-7B-Instruct-GGUF")
for f in files:
    if f.endswith(".gguf"):
        print(f)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


qwen2.5-7b-instruct-fp16-00001-of-00004.gguf
qwen2.5-7b-instruct-fp16-00002-of-00004.gguf
qwen2.5-7b-instruct-fp16-00003-of-00004.gguf
qwen2.5-7b-instruct-fp16-00004-of-00004.gguf
qwen2.5-7b-instruct-q2_k.gguf
qwen2.5-7b-instruct-q3_k_m.gguf
qwen2.5-7b-instruct-q4_0-00001-of-00002.gguf
qwen2.5-7b-instruct-q4_0-00002-of-00002.gguf
qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf
qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf
qwen2.5-7b-instruct-q5_0-00001-of-00002.gguf
qwen2.5-7b-instruct-q5_0-00002-of-00002.gguf
qwen2.5-7b-instruct-q5_k_m-00001-of-00002.gguf
qwen2.5-7b-instruct-q5_k_m-00002-of-00002.gguf
qwen2.5-7b-instruct-q6_k-00001-of-00002.gguf
qwen2.5-7b-instruct-q6_k-00002-of-00002.gguf
qwen2.5-7b-instruct-q8_0-00001-of-00003.gguf
qwen2.5-7b-instruct-q8_0-00002-of-00003.gguf
qwen2.5-7b-instruct-q8_0-00003-of-00003.gguf


In [ ]:
from huggingface_hub import hf_hub_download

repo_id = "Qwen/Qwen2.5-7B-Instruct-GGUF"

files = [
    "qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf",
    "qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf",
]

paths = []
for f in files:
    paths.append(
        hf_hub_download(repo_id=repo_id, filename=f)
    )

paths


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


['/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct-GGUF/snapshots/bb5d59e06d9551d752d08b292a50eb208b07ab1f/qwen2.5-7b-instruct-q4_k_m-00001-of-00002.gguf',
 '/root/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct-GGUF/snapshots/bb5d59e06d9551d752d08b292a50eb208b07ab1f/qwen2.5-7b-instruct-q4_k_m-00002-of-00002.gguf']

In [ ]:
!pip install transformers accelerate bitsandbytes
!pip uninstall -y transformers huggingface_hub
!pip install "transformers==4.43.0" "huggingface_hub==0.34.1" bitsandbytes

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Use GPU
device = "cuda"

# Prepare quantization config
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,              # quantize to 8-bit
    llm_int8_threshold=6.0,         # optional, default
    llm_int8_enable_fp32_cpu_offload=True  # offload layers that don't fit to CPU
)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# load model
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

# input
inputs = tokenizer("Hello world", return_tensors="pt")

# output in tokens
outputs = model.generate(**inputs, max_new_tokens=50)

# Decode
print(tokenizer.decode(outputs[0]))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

ImportError: Using `bitsandbytes` 8-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [ ]:
input